In [7]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()

In [8]:
spark = cc.startLocalCluster("DIM_CUSTOMER") # default 4
spark.getActiveSession()

In [9]:
cc.set_connectionProfile("VeloDB")
# todo incremantal laden bij customer en fact 
df_operational_users = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()


df_operational_subscription_type = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscription_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptiontypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_operational_subscriptions = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()


In [10]:
#TEMP VIEW
df_operational_users.createOrReplaceTempView("dimCustomer")
df_operational_subscription_type.createOrReplaceTempView("subscriptionTypes")
df_operational_subscriptions.createOrReplaceTempView("subscriptions")

In [11]:

# TRANSFORM: First, create a temp view for the most recent subscription per user
spark.sql("""

SELECT userId, validfrom, description
FROM (
SELECT 
        s.userId, 
        s.validfrom, 
        st.description,
        ROW_NUMBER() OVER (PARTITION BY s.userId ORDER BY s.validfrom DESC) AS rank
    FROM subscriptions AS s
    LEFT JOIN subscriptionTypes AS st ON s.subscriptiontypeid = st.subscriptiontypeid
)
WHERE rank = 1
""").createOrReplaceTempView("mostRecentSubscription")

# Final dimension: Join the customer and most recent subscription details
df_dim_customer = spark.sql("""
SELECT 
    monotonically_increasing_id() AS userSK, 
    c.userId, 
    c.name, 
    c.email, 
    CONCAT(c.street, ' ', c.number, ' ', c.zipcode, ' ', c.city, ' ',c.country_code) as address,
    m.validfrom, 
    m.description, 
    to_timestamp('1999-01-01','yyyy-MM-dd') AS scd_start, 
    to_timestamp('2100-12-12','yyyy-MM-dd') AS scd_end, 
    md5(concat(c.name, c.email, c.street, c.number, c.zipcode, c.city, c.country_code)) AS md5, 
    True AS current
FROM dimCustomer AS c
LEFT JOIN mostRecentSubscription AS m ON c.userId = m.userId
""")

# Create temp view for the final dimension
df_dim_customer.createOrReplaceTempView("dimCustomer")

# Display the final dimension
spark.sql("SELECT * FROM dimCustomer").show()

+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|     userSK|userId|                name|               email|             address| validfrom|description|          scd_start|            scd_end|                 md5|current|
+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|          0|     2|   van der Zee Julia|Julia.van.der.Zee...|Europalaan 43  26...|2021-10-05|      MAAND|1999-01-01 00:00:00|2100-12-12 00:00:00|01301120bda974125...|   true|
|          1|     4|      Willems Angela|Angela.Willems@te...|Graaf Joseph de P...|2019-04-08|        DAG|1999-01-01 00:00:00|2100-12-12 00:00:00|b0ce18a53c12438ee...|   true|
|          2|     1|         Bouman Lars|Lars.Bouman@gmail...|Somméstraat 156  ...|2020-12-14|        DAG|1999-01-01 00:

In [12]:
delta_table_path = "spark-warehouse/dimCustomer"
df_dim_customer.write.format("delta").mode("overwrite").save(delta_table_path)

In [13]:
df_dim_customer.createOrReplaceTempView("dimCustomer")
spark.sql("select * from dimCustomer").show()

+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|     userSK|userId|                name|               email|             address| validfrom|description|          scd_start|            scd_end|                 md5|current|
+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|          0|     2|   van der Zee Julia|Julia.van.der.Zee...|Europalaan 43  26...|2021-10-05|      MAAND|1999-01-01 00:00:00|2100-12-12 00:00:00|01301120bda974125...|   true|
|          1|     4|      Willems Angela|Angela.Willems@te...|Graaf Joseph de P...|2019-04-08|        DAG|1999-01-01 00:00:00|2100-12-12 00:00:00|b0ce18a53c12438ee...|   true|
|          2|     1|         Bouman Lars|Lars.Bouman@gmail...|Somméstraat 156  ...|2020-12-14|        DAG|1999-01-01 00:

In [17]:
df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("dimCustomers")
